# A.V.A.R.S Advanced ML: LSTM for Encrypted C2 Detection
## Deep Learning Model for Command & Control Traffic Analysis

**Objective**: Build an LSTM neural network that detects encrypted C2 (Command & Control) communication by analyzing **timing patterns**, not payload content.

### Why This Matters
- **C2 heartbeat problem**: Attackers send encrypted C2 traffic on regular schedules (e.g., every 30 seconds)
- **Pattern recognition**: Normal user traffic is bursty/irregular; C2 is metronomic (like a heartbeat)
- **Deep learning advantage**: LSTM can learn sequential timing patterns that traditional ML (Random Forest, Isolation Forest) cannot detect
- **Encrypted-agnostic**: Works on encrypted traffic where you cannot inspect payloads

### Key Concepts
| Concept | Explanation |
|---------|-------------|
| **LSTM** | Recurrent Neural Network layer that remembers sequences |
| **Sequence Length** | Look back 60 time steps (e.g., 60 seconds of traffic) |
| **Features** | bytes_sent, bytes_recv, packet_count, duration, entropy |
| **C2 Pattern** | Regular intervals, consistent size, metronomic timing |
| **Normal Traffic** | Irregular intervals, variable size, bursty pattern |

#### Environment
- **Framework**: TensorFlow/Keras
- **Libraries**: numpy, pandas, scikit-learn, matplotlib
- **GPU**: Optional (LSTM training 5-10x faster with GPU)

---

In [ ]:
# ============================================================================
# 1. IMPORT REQUIRED LIBRARIES & CONFIGURE ENVIRONMENT
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, 
    precision_recall_curve, f1_score, classification_report
)

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
)

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ Libraries imported successfully")
print(f"🔹 TensorFlow version: {tf.__version__}")
print(f"🔹 GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Configure GPU if available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"👉 GPU Memory Growth: Enabled for {len(gpus)} GPU(s)")
else:
    print("⚠️  No GPU detected. Training will use CPU (slower)")


## Section 1: Data Preparation & Feature Engineering

### Step 1: Generate Synthetic Traffic Data
- Normal traffic: Bursty, irregular timing, variable sizes
- C2 traffic: Metronomic, regular intervals, consistent sizes

### Step 2: Create Sliding Windows
- Convert 1D traffic sequence into 2D windows (time steps, features)
- Example: 60-second window = (60, 5) shape → fed to LSTM

### Step 3: Normalize Features
- Standardize all features to [0, 1] range
- Prevents large features from dominating learning


In [ ]:
# ============================================================================
# 2. GENERATE SYNTHETIC TRAFFIC DATA
# ============================================================================
# In production, this data would come from network captures (tcpdump, Sensor)

def generate_normal_traffic(n_samples=1000, duration_seconds=3600):
    """
    Generate NORMAL traffic pattern:
    - Bursty (random bursts)
    - Variable packet sizes
    - Irregular timing
    - High entropy (varied content)
    """
    np.random.seed(42)
    
    traffic = []
    for _ in range(n_samples):
        # Burst of traffic followed by silence (simulates human user behavior)
        if np.random.random() < 0.3:  # 30% chance of activity
            bytes_sent = np.random.exponential(scale=500)  # Variable size
            bytes_recv = np.random.exponential(scale=800)
            packets = np.random.randint(1, 50)
            duration = np.random.uniform(0.1, 5.0)  # Variable duration
            entropy = np.random.uniform(0.6, 1.0)  # High entropy (different payloads)
            is_c2 = 0
        else:
            # Idle period
            bytes_sent = np.random.normal(10, 5)
            bytes_recv = np.random.normal(10, 5)
            packets = np.random.randint(0, 3)
            duration = np.random.uniform(0.1, 1.0)
            entropy = np.random.uniform(0.1, 0.4)
            is_c2 = 0
        
        traffic.append([bytes_sent, bytes_recv, packets, duration, entropy, is_c2])
    
    return pd.DataFrame(traffic, columns=[
        'bytes_sent', 'bytes_recv', 'packets', 'duration', 'entropy', 'is_c2'
    ])

def generate_c2_traffic(n_samples=200, interval_seconds=30):
    """
    Generate C2 traffic pattern:
    - Metronomic (regular heartbeat)
    - Consistent packet sizes
    - Regular timing
    - Low entropy (encrypted payload)
    """
    np.random.seed(42)
    
    traffic = []
    for _ in range(n_samples):
        # Regular C2 heartbeat with small random variance
        bytes_sent = np.random.normal(200, 20)  # ~200 bytes, low variance
        bytes_recv = np.random.normal(100, 15)   # ~100 bytes, low variance
        packets = np.random.randint(2, 5)        # 2-4 packets per heartbeat
        duration = np.random.normal(0.5, 0.05)   # ~0.5 seconds, very consistent
        entropy = np.random.uniform(0.8, 1.0)    # High entropy (encrypted)
        is_c2 = 1
        
        traffic.append([bytes_sent, bytes_recv, packets, duration, entropy, is_c2])
    
    return pd.DataFrame(traffic, columns=[
        'bytes_sent', 'bytes_recv', 'packets', 'duration', 'entropy', 'is_c2'
    ])

# Generate datasets
print("📊 Generating synthetic traffic data...")
normal_traffic = generate_normal_traffic(n_samples=2000)
c2_traffic = generate_c2_traffic(n_samples=500)

# Combine and shuffle
all_traffic = pd.concat([normal_traffic, c2_traffic], ignore_index=True)
all_traffic = all_traffic.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Generated {len(all_traffic):,} traffic samples")
print(f"   - Normal traffic: {len(normal_traffic):,} ({len(normal_traffic)/len(all_traffic)*100:.1f}%)")
print(f"   - C2 traffic: {len(c2_traffic):,} ({len(c2_traffic)/len(all_traffic)*100:.1f}%)")
print(f"\n📋 Dataset Overview:")
print(all_traffic.describe())

# Visualize traffic patterns
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Traffic Patterns: Normal vs C2', fontsize=16, fontweight='bold')

features = ['bytes_sent', 'bytes_recv', 'packets', 'duration', 'entropy']
for idx, feature in enumerate(features):
    row = idx // 3
    col = idx % 3
    
    axes[row, col].hist(normal_traffic[feature], bins=30, alpha=0.6, label='Normal', color='blue')
    axes[row, col].hist(c2_traffic[feature], bins=30, alpha=0.6, label='C2', color='red')
    axes[row, col].set_title(f'{feature}')
    axes[row, col].set_xlabel('Value')
    axes[row, col].set_ylabel('Frequency')
    axes[row, col].legend()
    axes[row, col].grid(alpha=0.3)

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("✅ Data generation complete")


In [ ]:
# ============================================================================
# 3. CREATE SEQUENCES FOR LSTM (Sliding Window)
# ============================================================================

class SequenceGenerator:
    """
    Convert 1D time series data into 2D sequences for LSTM
    
    Example:
        If sequence_length=60, creates windows of 60 consecutive time steps
        Shape: (n_sequences, 60, n_features)
    """
    
    def __init__(self, sequence_length=60):
        self.sequence_length = sequence_length
        self.scaler = StandardScaler()
    
    def prepare_sequences(self, data, labels=None):
        """
        Convert flat traffic data into sliding window sequences
        
        Args:
            data: DataFrame or numpy array with shape (n_samples, n_features)
            labels: Optional binary labels (0=normal, 1=C2)
        
        Returns:
            X: (n_sequences, sequence_length, n_features) array
            y: Optional labels for each sequence
        """
        
        # Select feature columns only (exclude 'is_c2')
        if isinstance(data, pd.DataFrame):
            feature_cols = data.columns[data.columns != 'is_c2']
            features = data[feature_cols].values
            if labels is None and 'is_c2' in data.columns:
                labels = data['is_c2'].values
        else:
            features = data
        
        # Normalize features to [0, 1]
        features = self.scaler.fit_transform(features)
        
        X = []
        y_list = []
        
        # Create sliding windows
        for i in range(len(features) - self.sequence_length):
            window = features[i:i + self.sequence_length]
            X.append(window)
            
            if labels is not None:
                # Label sequence based on majority class in window
                window_labels = labels[i:i + self.sequence_length]
                y_list.append(1 if np.mean(window_labels) > 0.5 else 0)
        
        X = np.array(X)
        
        if labels is not None:
            return X, np.array(y_list)
        
        return X
    
    def transform(self, data):
        """Transform data using fitted scaler"""
        if isinstance(data, pd.DataFrame):
            feature_cols = data.columns[data.columns != 'is_c2']
            features = data[feature_cols].values
        else:
            features = data
        
        return self.scaler.transform(features)

# Create sequences
print("🔄 Creating sliding window sequences...")
seq_gen = SequenceGenerator(sequence_length=60)

# Prepare training sequences
X_sequences, y_sequences = seq_gen.prepare_sequences(all_traffic)

print(f"✅ Sequences created:")
print(f"   - Shape: {X_sequences.shape}")
print(f"   - Time steps: {X_sequences.shape[1]}")
print(f"   - Features per step: {X_sequences.shape[2]}")
print(f"   - C2 sequences: {np.sum(y_sequences)} ({np.sum(y_sequences)/len(y_sequences)*100:.1f}%)")
print(f"   - Normal sequences: {len(y_sequences) - np.sum(y_sequences)}")

# Visualize sample sequence
fig, ax = plt.subplots(figsize=(14, 6))

# Plot a normal sequence example
normal_idx = np.where(y_sequences == 0)[0][0]
c2_idx = np.where(y_sequences == 1)[0][0]

for i, feature_idx in enumerate([0, 1, 3]):  # bytes_sent, bytes_recv, duration
    ax.plot(X_sequences[normal_idx, :, feature_idx], alpha=0.7, label=f'Normal - Feature {feature_idx}')
    ax.plot(X_sequences[c2_idx, :, feature_idx] + 2, alpha=0.7, linestyle='--', label=f'C2 - Feature {feature_idx}')

ax.set_title('Sequence Pattern: Normal vs C2 Traffic (60-second window)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Steps')
ax.set_ylabel('Normalized Feature Value')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sequences, y_sequences,
    test_size=0.2,
    random_state=42,
    stratify=y_sequences  # Maintain class balance
)

print(f"\n📊 Train/Test Split:")
print(f"   - Training: {len(X_train)} sequences")
print(f"   - Testing: {len(X_test)} sequences")
print(f"   - Class balance (train): {np.sum(y_train)/len(y_train)*100:.1f}% C2")
print(f"   - Class balance (test): {np.sum(y_test)/len(y_test)*100:.1f}% C2")


## Section 2: LSTM Neural Network Architecture

### Model Design

```
Input Layer (60, 5)
    ↓
LSTM Layer 1 (128 units)
    ↓
Dropout (20%)
    ↓
LSTM Layer 2 (64 units)
    ↓
Dropout (20%)
    ↓
LSTM Layer 3 (32 units)
    ↓
Dense Layer (16 units, ReLU)
    ↓
Dropout (10%)
    ↓
Dense Layer (8 units, ReLU)
    ↓
Output Layer (1 unit, Sigmoid) → C2 Probability
```

### Key Design Decisions

| Component | Choice | Reason |
|-----------|--------|--------|
| **Layers** | 3 LSTM layers | Capture multi-level temporal patterns |
| **Units** | 128 → 64 → 32 | Hierarchical feature extraction |
| **Activation** | ReLU | Non-linearity for complex patterns |
| **Dropout** | 20% + 10% | Prevent overfitting |
| **Output** | Sigmoid | Binary classification (0-1 probability) |
| **Loss** | Binary Crossentropy | Standard for binary classification |
| **Optimizer** | Adam | Adaptive learning rate |


In [ ]:
# ============================================================================
# 4. BUILD LSTM MODEL
# ============================================================================

def build_lstm_model(input_shape, learning_rate=0.001):
    """
    Build LSTM neural network for C2 detection
    
    Args:
        input_shape: (sequence_length, n_features) e.g., (60, 5)
        learning_rate: Adam optimizer learning rate
    
    Returns:
        Compiled Keras model
    """
    
    model = models.Sequential([
        # Layer 1: LSTM with 128 units (longer-term patterns)
        layers.LSTM(
            units=128,
            activation='relu',
            input_shape=input_shape,
            return_sequences=True,  # Pass sequences to next layer
            name='lstm_1'
        ),
        layers.Dropout(0.2, name='dropout_1'),
        
        # Layer 2: LSTM with 64 units (mid-term patterns)
        layers.LSTM(
            units=64,
            activation='relu',
            return_sequences=True,
            name='lstm_2'
        ),
        layers.Dropout(0.2, name='dropout_2'),
        
        # Layer 3: LSTM with 32 units (short-term patterns)
        layers.LSTM(
            units=32,
            activation='relu',
            return_sequences=False,  # Return only final output
            name='lstm_3'
        ),
        
        # Dense layers for feature processing
        layers.Dense(units=16, activation='relu', name='dense_1'),
        layers.Dropout(0.1, name='dropout_3'),
        
        layers.Dense(units=8, activation='relu', name='dense_2'),
        
        # Output layer: Binary classification
        layers.Dense(units=1, activation='sigmoid', name='output')
    ], name='LSTM_C2_Detector')
    
    # Compile model
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    
    return model

# Build model
print("🏗️  Building LSTM model...")
model = build_lstm_model(input_shape=(X_train.shape[1], X_train.shape[2]))

print("✅ Model built successfully!")
print(f"📊 Model Summary:")
model.summary()

# Visualize model architecture
print("\n🔍 Model Parameters:")
print(f"   - Total parameters: {model.count_params():,}")
print(f"   - Trainable parameters: {sum(w.numpy().size for w in model.trainable_weights):,}")


## Section 3: Training the LSTM Model

### Training Strategy

1. **Early Stopping**: Halt training if validation loss stops improving
2. **Model Checkpoint**: Save best weights
3. **Learning Rate Reduction**: Reduce LR if validation loss plateaus
4. **Validation Split**: 20% of training data for validation monitoring

### Expected Results
- Binary crossentropy loss: < 0.3
- AUC score: > 0.90 (0.95+ is excellent)
- Precision/Recall balance achieved


In [ ]:
# ============================================================================
# 5. TRAIN LSTM MODEL
# ============================================================================

# Callbacks
callbacks = [
    # Stop training if validation loss doesn't improve
    EarlyStopping(
        monitor='val_loss',
        patience=10,  # Stop after 10 epochs of no improvement
        restore_best_weights=True,
        verbose=1
    ),
    
    # Save best model weights
    ModelCheckpoint(
        filepath='model_checkpoint.h5',
        monitor='val_auc',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce learning rate if validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    
    # TensorBoard logging
    TensorBoard(
        log_dir='./logs',
        histogram_freq=1,
        write_graph=True
    )
]

# Train model
print("🚀 Training LSTM model...")
print(f"   - Training samples: {len(X_train):,}")
print(f"   - Batch size: 32")
print(f"   - Epochs: 100 (with early stopping)")
print()

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
    class_weight={0: 1.0, 1: 3.0}  # Weight C2 class more heavily
)

print("\n✅ Training complete!")

# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LSTM Model Training History', fontsize=16, fontweight='bold')

# Loss
axes[0, 0].plot(history.history['loss'], label='Training', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_title('Loss over Epochs')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Binary Crossentropy')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Training', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 1].set_title('Accuracy over Epochs')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# AUC
axes[1, 0].plot(history.history['auc'], label='Training', linewidth=2)
axes[1, 0].plot(history.history['val_auc'], label='Validation', linewidth=2)
axes[1, 0].set_title('AUC over Epochs')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUC')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Precision & Recall
axes[1, 1].plot(history.history['precision'], label='Precision (Train)', linewidth=2)
axes[1, 1].plot(history.history['recall'], label='Recall (Train)', linewidth=2)
axes[1, 1].plot(history.history['val_precision'], label='Precision (Val)', linewidth=2, linestyle='--')
axes[1, 1].plot(history.history['val_recall'], label='Recall (Val)', linewidth=2, linestyle='--')
axes[1, 1].set_title('Precision & Recall over Epochs')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Final metrics
final_loss = history.history['val_loss'][-1]
final_auc = history.history['val_auc'][-1]
final_accuracy = history.history['val_accuracy'][-1]

print(f"\n📈 Final Training Metrics (Validation Set):")
print(f"   - Loss: {final_loss:.4f}")
print(f"   - Accuracy: {final_accuracy:.4f}")
print(f"   - AUC: {final_auc:.4f}")


## Section 4: Model Evaluation on Test Set

### Test Set Metrics
- **Accuracy**: Percentage of correct predictions
- **Precision**: Of detected C2, how many are actually C2? (False positive rate)
- **Recall**: Of actual C2, how many did we detect? (False negative rate)
- **F1 Score**: Harmonic mean of precision & recall
- **ROC-AUC**: Area under receiver operating characteristic curve (>0.90 is excellent)


In [ ]:
# ============================================================================
# 6. EVALUATE MODEL ON TEST SET
# ============================================================================

# Get predictions on test set
print("📊 Evaluating model on test set...")
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob.flatten() > 0.5).astype(int)

# Calculate metrics
test_loss, test_accuracy, test_auc, test_precision, test_recall = model.evaluate(
    X_test, y_test, verbose=0
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Additional metrics
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print(f"\n📈 TEST SET RESULTS")
print(f"{'='*50}")
print(f"Accuracy:     {test_accuracy:.4f}")
print(f"Precision:    {test_precision:.4f}  (Low false positives)")
print(f"Recall:       {test_recall:.4f}  (Detect C2 attacks)")
print(f"F1 Score:     {f1:.4f}")
print(f"ROC-AUC:      {roc_auc:.4f}  {'✅ Excellent' if roc_auc > 0.95 else '✅ Good' if roc_auc > 0.90 else '⚠️  Fair'}")
print(f"{'='*50}")

print(f"\n🎯 CONFUSION MATRIX")
print(f"{'='*50}")
print(f"True Negatives:   {tn:4d}  (Correctly identified normal)")
print(f"False Positives:  {fp:4d}  (False alarms)")
print(f"False Negatives:  {fn:4d}  (Missed C2 attacks)")
print(f"True Positives:   {tp:4d}  (Correctly detected C2)")
print(f"{'='*50}")

# Specificity and sensitivity
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"\n🔍 ADDITIONAL METRICS")
print(f"Sensitivity (Recall):  {sensitivity:.4f}  (Catch C2 attacks)")
print(f"Specificity:           {specificity:.4f}  (Avoid false alarms)")

# Visualize confusion matrix
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('LSTM Model Evaluation on Test Set', fontsize=16, fontweight='bold')

# Confusion matrix heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=axes[0, 0],
    cbar=False,
    xticklabels=['Normal', 'C2'],
    yticklabels=['Normal', 'C2']
)
axes[0, 0].set_title('Confusion Matrix')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
axes[0, 1].plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC={roc_auc:.4f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Precision-Recall Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_prob)
axes[1, 0].plot(recall_vals, precision_vals, linewidth=2, color='green')
axes[1, 0].set_xlabel('Recall (True Positive Rate)')
axes[1, 0].set_ylabel('Precision (1 - False Positive Rate)')
axes[1, 0].set_title('Precision-Recall Curve')
axes[1, 0].grid(alpha=0.3)

# Classification metrics bar chart
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
metrics_values = [test_accuracy, test_precision, test_recall, f1, roc_auc]
colors = ['green' if v > 0.9 else 'orange' if v > 0.8 else 'red' for v in metrics_values]

axes[1, 1].bar(metrics_names, metrics_values, color=colors, alpha=0.7)
axes[1, 1].set_ylim([0, 1])
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('Model Performance Metrics')
axes[1, 1].axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='Excellent (>0.9)')
axes[1, 1].axhline(y=0.8, color='orange', linestyle='--', alpha=0.5, label='Good (>0.8)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(axis='y', alpha=0.3)
axes[1, 1].legend()

# Add value labels on bars
for i, v in enumerate(metrics_values):
    axes[1, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Classification report
print(f"\n📋 CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred, target_names=['Normal', 'C2']))


## Section 5: Model Interpretability & Real-World Deployment

### Explain Model Decisions
- **Feature importance**: Which features matter most?
- **Decision boundaries**: Why does model classify as C2?
- **Confidence levels**: How certain is the model?

### Production Deployment Strategy
1. **Save model**: Save to disk/cloud storage
2. **Create inference API**: Flask/FastAPI for real-time predictions
3. **Monitor drift**: Retrain when data distribution changes
4. **Integrate with SIEM**: Feed predictions to Sentinel


In [ ]:
# ============================================================================
# 7. SAVE MODEL FOR DEPLOYMENT
# ============================================================================

import json
import os

# Create models directory
os.makedirs('./models', exist_ok=True)

# Save trained model
model_path = './models/lstm_c2_detector.h5'
model.save(model_path)
print(f"✅ Model saved to: {model_path}")

# Save model metadata for documentation
metadata = {
    'model_name': 'LSTM C2 Detector',
    'model_type': 'LSTM (3 layers)',
    'sequence_length': 60,
    'features': ['bytes_sent', 'bytes_recv', 'packets', 'duration', 'entropy'],
    'n_parameters': int(model.count_params()),
    'training_samples': int(len(X_train)),
    'test_samples': int(len(X_test)),
    'metrics': {
        'test_accuracy': float(test_accuracy),
        'test_precision': float(test_precision),
        'test_recall': float(test_recall),
        'test_f1_score': float(f1),
        'test_roc_auc': float(roc_auc),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity)
    },
    'training_config': {
        'optimizer': 'Adam',
        'learning_rate': 0.001,
        'loss': 'Binary Crossentropy',
        'batch_size': 32,
        'epochs_trained': len(history.history['loss'])
    },
    'deployment_notes': 'Ready for production. Expects normalized traffic features.'
}

metadata_path = './models/lstm_c2_detector_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata saved to: {metadata_path}")
print(f"\n📊 Model Metadata:")
print(json.dumps(metadata, indent=2))

# Example: Real-time prediction function
print("\n" + "="*70)
print("🚀 PRODUCTION INFERENCE EXAMPLE")
print("="*70)

def predict_c2_probability(traffic_sequence):
    """
    Predict probability that a traffic sequence is C2
    
    Args:
        traffic_sequence: numpy array of shape (60, 5)
    
    Returns:
        float: Probability (0-1) that traffic is C2
    """
    # Reshape for batch prediction
    sequence = np.expand_dims(traffic_sequence, axis=0)
    
    # Get prediction
    probability = float(model.predict(sequence, verbose=0)[0][0])
    
    # Determine if C2
    is_c2 = probability > 0.5
    
    return {
        'c2_probability': probability,
        'is_c2': is_c2,
        'confidence': max(probability, 1 - probability),
        'alert_level': (
            'CRITICAL' if probability > 0.9 else
            'HIGH' if probability > 0.7 else
            'MEDIUM' if probability > 0.5 else
            'LOW'
        )
    }

# Test on sample sequences
print("\n📋 Sample Predictions:")
print(f"{'Sequence':<15} {'C2_Prob':<12} {'Is_C2':<10} {'Confidence':<12} {'Alert':<12}")
print("-" * 60)

for i in range(5):
    result = predict_c2_probability(X_test[i])
    seq_type = 'C2' if y_test[i] == 1 else 'Normal'
    
    print(f"{seq_type:<15} "
          f"{result['c2_probability']:<12.4f} "
          f"{str(result['is_c2']):<10} "
          f"{result['confidence']:<12.4f} "
          f"{result['alert_level']:<12}")

print("\n✅ Model ready for deployment in production!")
print(f"\n💡 Next Steps:")
print(f"   1. Deploy model to Azure ML or containerized service")
print(f"   2. Create inference API (Flask/FastAPI)")
print(f"   3. Integrate with Sentinel alert rules")
print(f"   4. Monitor model drift and retrain quarterly")
print(f"   5. Set up alerts for confidence > 0.8")
